[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%20-%20NLP/13_LLMs.ipynb)


# LLMs — capacidades cognitivas con Gemini

En el **12** recuperaste páginas con embeddings + FAISS. Aquí el foco es el **modelo de lenguaje** (LLM): cómo hablarle de forma estructurada y qué más puede hacer (razonar, ver imágenes, usar herramientas).

**Idea central de este cuaderno:** un LLM no recibe solo “un texto suelto”. Recibe una **conversación** con **roles** — sobre todo el **prompt de sistema**, los mensajes del **usuario** y las respuestas previas del **modelo**.

Usamos la **Gemini API** (`google-genai`) y la misma clave `.env` que en el **11** y **12**.

| Bloque | Tema |
|--------|------|
| 2–5 | **Roles y prompts** (lo más importante) |
| 6 | Tokens |
| 7 | Razonamiento (*thinking*) |
| 8–9 | Visión |
| 10 | Function calling |
| 11 | Búsqueda en Google |
| 12 | JSON estructurado |


## Setup — instalar librerías

- `google-genai`, `python-dotenv`, `requests`, `pypdf` (para leer el expediente en la sección 9)


In [ ]:
# Comentar esta linea si etás en local 
%pip install -q google-genai python-dotenv requests pypdf


In [ ]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader

BASE_DIR = Path(".")


## 1 — Clave API (igual que notebooks 11 y 12)

Archivo `.env` en esta carpeta con `GOOGLE_API_KEY=...` ([AI Studio](https://aistudio.google.com/apikey)).


In [ ]:
# Local
# load_dotenv(BASE_DIR / ".env", override=True)
#
# API_KEY = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
# if not API_KEY:
#     raise ValueError("Falta GOOGLE_API_KEY en .env — https://aistudio.google.com/apikey")
#
# cliente = genai.Client(api_key=API_KEY)
# MODELO = "gemini-2.5-flash"
# MODELO_THINKING = "gemini-2.5-flash"
#
# print(f"Cliente listo. Modelo: {MODELO}")


In [ ]:
# Google Colab: descomenta si no usas .env
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

cliente = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODELO = "gemini-2.5-flash"
MODELO_THINKING = "gemini-2.5-flash"

print(f"Cliente listo. Modelo: {MODELO}")


In [ ]:
def mostrar(respuesta, titulo=None):
    if titulo:
        print(f"=== {titulo} ===\n")
    print(respuesta.text)
    u = getattr(respuesta, "usage_metadata", None)
    if u:
        print(
            f"\nTokens — entrada: {u.prompt_token_count}, "
            f"salida: {u.candidates_token_count}, total: {u.total_token_count}"
        )


def imprimir_historial(historial):
    """Muestra en consola la lista de mensajes que mandamos a la API."""
    for i, msg in enumerate(historial, start=1):
        texto = "".join(p.text for p in msg.parts if getattr(p, "text", None))
        print(f"{i}. [{msg.role}] {texto[:120]}{'…' if len(texto) > 120 else ''}")


## 2 — ¿Qué es un “prompt” en un LLM?

Cuando escribes en ChatGPT o en un chatbot, en realidad estás armando **mensajes con rol**. La API de Gemini recibe lo mismo: una lista ordenada de turnos.

| Rol | Nombre en Gemini | ¿Quién lo escribe? | Para qué sirve |
|-----|------------------|--------------------|----------------|
| **Sistema** | `system_instruction` (va aparte del historial) | Tú (desarrollador) | Reglas fijas: tono, personalidad, límites, formato. El usuario **no** lo ve en el chat, pero el modelo **sí** lo lee en cada llamada. |
| **Usuario** | `role="user"` en `Content` | La persona que pregunta | La duda, el dato, la imagen, la orden del momento. |
| **Modelo** | `role="model"` en `Content` | El LLM (turnos anteriores) | Respuestas que ya dio el asistente. Sirven para **continuar** la conversación con contexto. |

**Analogía rápida**

- **Sistema** = manual de operación del bot (“eres Iber.ia, habla en español, no inventes horarios”).
- **Usuario** = lo que pregunta el alumno hoy.
- **Modelo** = lo que el bot ya contestó antes en la misma sesión.

Si solo mandas un string (`contents="Hola"`), la API lo trata como **un único mensaje de usuario** sin system ni historial. Funciona para pruebas; en producción casi siempre querrás definir **system + historial**.


### 2.1 — Un solo mensaje de usuario (sin system)

Es lo más mínimo: una pregunta y una respuesta.


In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Hola! ¿Cómo estás?",
)
mostrar(respuesta)


### 2.2 — El mismo mensaje, pero explícito

Equivalente al anterior: declaramos el rol `user` con `types.Content`.


In [ ]:
mensaje_usuario = types.Content(
    role="user",
    parts=[types.Part.from_text(text="Hola! ¿Cómo estás?")],
)

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[mensaje_usuario],
)
mostrar(respuesta, "Content con role=user")


### 2.3 — Prompt de **sistema** (`system_instruction`)

No es un mensaje del chat: es el **manual del asistente**. Va en `system_instruction` y el modelo lo lee en cada llamada.

Un buen prompt de sistema suele incluir:

- **Objetivo** — para qué existe el bot
- **Tareas** — qué puede hacer (y qué no)
- **Tono** — cómo debe hablar
- **Límites** — qué debe evitar (inventar datos, dar consejo médico, etc.)


In [ ]:
SYSTEM_IBERIA = """
Eres Iber.ia, asistente virtual de la Universidad Iberoamericana Ciudad de México.

OBJETIVO
Ayudar a alumnos con dudas académicas y administrativas de forma clara y confiable.

TAREAS
- Responder preguntas sobre inscripción, horarios, biblioteca, servicios escolares y trámites comunes.
- Orientar al alumno: si falta un dato, pide contexto (carrera, semestre, campus).
- Si no sabes algo con certeza, dilo y sugiere contactar la mesa de ayuda o el departamento correspondiente.

TONO
Español, cercano pero profesional. Respuestas breves (3–6 oraciones salvo que pidan detalle).

LÍMITES
- No inventes fechas, costos, reglamentos ni disponibilidad de materias.
- No des consejo médico, legal ni psicológico.
- No compartas datos personales de otras personas.
"""

pregunta = "¿Quién eres y en qué me puedes ayudar?"

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[types.Content(role="user", parts=[types.Part.from_text(text=pregunta)])],
    config=types.GenerateContentConfig(system_instruction=SYSTEM_IBERIA),
)
mostrar(respuesta, "Con system_instruction")


**Conclusión:** el mensaje del usuario es corto; el **system** define identidad, alcance y reglas. En la sección 2.4 reutilizamos el mismo `SYSTEM_IBERIA` para el historial multi-turno.


### 2.4 — Historial: usuario + modelo (multi-turno)

Un chat real no es un solo par pregunta–respuesta. Envías **toda la conversación anterior** para que el modelo entienda el contexto.

En Gemini las respuestas del asistente van con rol **`model`** (así se nombra en la API de Google).

Construimos el caso **Iber.ia** paso a paso.


**Turno 1 — solo el saludo del usuario** (aún no hay respuestas del modelo en el historial).


In [ ]:
# Reutilizamos SYSTEM_IBERIA definido en 2.3
config_iberia = types.GenerateContentConfig(system_instruction=SYSTEM_IBERIA)

historial = [
    types.Content(role="user", parts=[types.Part.from_text(text="Hola!")]),
]

imprimir_historial(historial)

respuesta_1 = cliente.models.generate_content(
    model=MODELO,
    contents=historial,
    config=config_iberia,
)
mostrar(respuesta_1, "Turno 1")


**Turno 2 — agregamos la respuesta del modelo y el siguiente mensaje del usuario**

Patrón típico en código:

1. Mandas `historial` a la API.
2. Recibes `respuesta.text`.
3. Añades al historial: primero el `model`, luego el nuevo `user`.
4. Vuelves a llamar a la API con el historial actualizado.


In [ ]:
# Lo que el modelo acaba de decir pasa a ser un mensaje role=model
historial.append(
    types.Content(
        role="model",
        parts=[types.Part.from_text(text=respuesta_1.text)],
    )
)
# Nuevo mensaje del alumno
historial.append(
    types.Content(role="user", parts=[types.Part.from_text(text="Me llamo Emilio, ¿tú cómo te llamas?")])
)

print("Historial completo que enviaremos:\n")
imprimir_historial(historial)

respuesta_2 = cliente.models.generate_content(
    model=MODELO,
    contents=historial,
    config=config_iberia,
)
mostrar(respuesta_2, "Turno 2")


**Turno 3 — otra pregunta en la misma conversación**

No hace falta una función especial: repites el mismo patrón — guardas la respuesta del modelo, añades el nuevo `user`, vuelves a llamar a la API.


In [ ]:
historial.append(
    types.Content(role="model", parts=[types.Part.from_text(text=respuesta_2.text)])
)
historial.append(
    types.Content(role="user", parts=[types.Part.from_text(text="¿Dónde está la biblioteca central?")])
)

imprimir_historial(historial)

respuesta_3 = cliente.models.generate_content(
    model=MODELO,
    contents=historial,
    config=config_iberia,
)
mostrar(respuesta_3, "Turno 3")


## 3 — Tokens

Cada llamada consume tokens de **entrada** (system + historial + pregunta) y de **salida** (respuesta). Revisa `usage_metadata` para estimar costo.


In [ ]:
u = respuesta_3.usage_metadata
print(f"Entrada:  {u.prompt_token_count}")
print(f"Salida:   {u.candidates_token_count}")
print(f"Total:    {u.total_token_count}")


## 4 — Razonamiento (*thinking*)

Gemini 2.5 puede gastar tokens **internos** de razonamiento antes de escribir la respuesta. Controlas cuánto con `thinking_budget` (más presupuesto → problemas más difíciles).

> El alumno normalmente solo ve la respuesta final, no el pensamiento interno.


## 5 — Visión: imagen en internet

El mensaje del **usuario** puede llevar **texto + imagen** en `parts`.

> **Ojo:** sitios como Wikimedia exigen `User-Agent` en el `requests.get`. Sin eso devuelven **403** (texto, no JPEG) y Gemini responde `Unable to process input image`.


In [ ]:
URL_IMAGEN = "https://upload.wikimedia.org/wikipedia/commons/d/da/The_Parthenon_in_Athens.jpg"

resp = requests.get(
    URL_IMAGEN,
    timeout=60,
    headers={"User-Agent": "ibero-nlp-notebook/1.0 (uso educativo)"},
)
resp.raise_for_status()
bytes_img = resp.content

if not bytes_img.startswith(b"\xff\xd8\xff"):
    raise ValueError(f"No llegó un JPEG válido (status {resp.status_code}, {len(bytes_img)} bytes)")

print(f"Imagen descargada: {len(bytes_img) / 1024:.0f} KB")

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_bytes(data=bytes_img, mime_type="image/jpeg"),
                types.Part.from_text(text="Describe en español qué ves (2–4 oraciones)."),
            ],
        )
    ],
)
mostrar(respuesta, "Visión — URL")


## 7 — Function calling

El **usuario** pide el clima; el **modelo** decide invocar tu función `get_weather`. La librería puede encadenar la llamada sola (`automatic_function_calling`).


In [ ]:
def get_weather(latitude: float, longitude: float) -> dict:
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    temp = requests.get(url, timeout=30).json()["current"]["temperature_2m"]
    return {"latitude": latitude, "longitude": longitude, "temperature_c": temp}

print(get_weather(19.29, -99.18))


In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="¿Temperatura aproximada en Tlalpan, CDMX? Usa coordenadas razonables.",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=3),
    ),
)
mostrar(respuesta, "Clima")


## 8 — Búsqueda en Google (grounding)

El modelo puede **buscar en la web** y citar fuentes. Documentación: [Google Search grounding](https://ai.google.dev/gemini-api/docs/google-search).


In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Menciona 3 hechos recientes sobre adopción de IA en empresas mexicanas. Cita fuentes si puedes.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
    ),
)
mostrar(respuesta, "Grounding")


## 9 — Respuesta estructurada (JSON): expediente médico

En lugar de parsear a mano un PDF, le pasamos el **texto** del documento al LLM y pedimos salida **JSON** con campos fijos (`response_schema`).

Archivo de ejemplo en esta carpeta: `expediente_Rivera_Jose.pdf` (Hospital Estrella del Norte).


### 9.1 — Leer el PDF a texto


In [ ]:
RUTA_EXPEDIENTE = BASE_DIR / "expediente_Rivera_Jose.pdf"
if not RUTA_EXPEDIENTE.exists():
    raise FileNotFoundError(f"No encontré {RUTA_EXPEDIENTE.name}")

reader = PdfReader(RUTA_EXPEDIENTE)
texto_expediente = "\n\n".join(
    (page.extract_text() or "").strip() for page in reader.pages
)

print(f"Páginas: {len(reader.pages)}  |  Caracteres: {len(texto_expediente)}")
print(texto_expediente[:600], "...\n")


### 9.2 — Extraer campos con esquema JSON

Definimos qué campos queremos. En **Diagnóstico** del expediente hay varias enfermedades en un párrafo; pedimos **una fila por diagnóstico** en un arreglo `diagnosticos` (como filas de una tabla).


In [ ]:
esquema_expediente = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "nombre_completo": types.Schema(type=types.Type.STRING),
        "curp": types.Schema(type=types.Type.STRING),
        "fecha_nacimiento": types.Schema(type=types.Type.STRING),
        "edad": types.Schema(type=types.Type.INTEGER),
        "sexo": types.Schema(type=types.Type.STRING),
        "tipo_sangre": types.Schema(type=types.Type.STRING),
        "telefono": types.Schema(type=types.Type.STRING),
        "alergias": types.Schema(type=types.Type.STRING),
        "enfermedades_cronicas": types.Schema(type=types.Type.STRING),
        "medicamentos_actuales": types.Schema(type=types.Type.STRING),
        "ultima_visita": types.Schema(
            type=types.Type.OBJECT,
            description="Visita con fecha más reciente del expediente",
            properties={
                "fecha": types.Schema(type=types.Type.STRING),
                "motivo": types.Schema(type=types.Type.STRING),
                "diagnosticos": types.Schema(
                    type=types.Type.ARRAY,
                    description="Un elemento por cada diagnóstico (una fila c/u)",
                    items=types.Schema(type=types.Type.STRING),
                ),
            },
            required=["fecha", "diagnosticos"],
        ),
    },
    required=[
        "nombre_completo",
        "curp",
        "alergias",
        "enfermedades_cronicas",
        "medicamentos_actuales",
        "ultima_visita",
    ],
)

prompt_extraccion = f"""
Extrae la información del siguiente expediente médico.
Usa solo lo que aparece en el texto; si un campo no está, usa null o cadena vacía.

Para ultima_visita usa la visita con fecha más reciente.
En diagnosticos: separa cada diagnóstico en un elemento distinto del arreglo
(no los juntes en un solo párrafo). Ejemplo de formato (valores inventados):
- Rinofaringitis aguda
- Lumbalgia mecánica inespecífica
- Deficiencia de vitamina D

--- EXPEDIENTE ---
{texto_expediente}
--- FIN ---
"""

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=prompt_extraccion,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=esquema_expediente,
    ),
)

datos_paciente = json.loads(respuesta.text)
print(json.dumps(datos_paciente, indent=2, ensure_ascii=False))

print("\n--- Diagnósticos (una fila c/u) ---")
for i, dx in enumerate(datos_paciente["ultima_visita"]["diagnosticos"], start=1):
    print(f"{i}. {dx}")


## 10 — Cierre

| Ya viste | Siguiente paso |
|----------|----------------|
| **System + user + model** | Diseñar el system de tu chatbot (tono, límites, citas) |
| **Historial** | Guardar sesión en lista o base de datos |
| **Visión / tools / JSON** | Tickets, clima, **expedientes médicos** en JSON |
| **Notebook 12 (RAG)** | Recuperar páginas con FAISS y pasar el texto al LLM como mensaje **user** para la respuesta final con evidencia |

Eso es un asistente completo: **recuperar** (12) + **razonar y redactar** (este cuaderno).
